# EA Sports Player Performance Index


## Import libraries

In [1]:
import math
import sys
import warnings
from pathlib import Path

import pandas as pd
from kloppy import statsbomb, wyscout

In [2]:
# Add the project root to the Python path
sys.path.append(str(Path.cwd().parents[1]))
from config import project_paths

In [3]:
# Ignore future warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings(
    "ignore",
    message="Boolean Series key will be reindexed to match DataFrame index.",
    category=UserWarning,
)

## Load providers data

In [4]:
# StatsBomb match and player IDs for testing (UEFA Euro 2024 Final - Lamine Yamal)
MATCH_ID = 3943043
PLAYER_ID = 316046
PROVIDER = "statsbomb"

In [5]:
# Load dataset based on provider
if PROVIDER == "statsbomb":
    dataset = statsbomb.load(
        event_data=project_paths.STATSBOMB_EVENTS_DIR / f"{MATCH_ID}.json",
        lineup_data=project_paths.STATSBOMB_LINEUPS_DIR / f"{MATCH_ID}.json",
    )
elif PROVIDER == "wyscout":
    dataset = wyscout.load(
        event_data=project_paths.WYSCOUT_PROCESSED_V2_DIR / f"{MATCH_ID}.json",
    )
else:
    raise ValueError("Unsupported provider. Use 'statsbomb' or 'wyscout'.")

## Data exploration

In [6]:
# Convert dataset to DataFrame for easier exploration
df = dataset.to_df()

In [7]:
# Display the first 5 rows of the dataset
df.head()

,event_id,event_type,period_id,timestamp,end_timestamp,ball_state,ball_owning_team,team_id,player_id,coordinates_x,...,end_coordinates_y,receiver_player_id,set_piece_type,body_part_type,pass_type,is_under_pressure,duel_type,is_counter_attack,goalkeeper_type,card_type
0,50aa204f-5d65-4145-8597-5d5628fb7898,GENERIC:Starting XI,1,0 days 00:00:00,NaT,alive,772,772,None,NaN,...,NaN,None,None,None,None,None,None,None,None,None
1,a279cbee-9ab3-4cfb-9c51-27cacc1bf2a2,GENERIC:Starting XI,1,0 days 00:00:00,NaT,alive,772,768,None,NaN,...,NaN,None,None,None,None,None,None,None,None,None
2,d2126e70-9f04-4bb7-ba2b-9377836d1757,GENERIC:Half Start,1,0 days 00:00:00,NaT,alive,772,768,None,NaN,...,NaN,None,None,None,None,None,None,None,None,None
3,54d78bfa-4146-42bd-acdc-97bcd393dd81,GENERIC:Half Start,1,0 days 00:00:00,NaT,alive,772,772,None,NaN,...,NaN,None,None,None,None,None,None,None,None,None
4,152820f0-6ca9-4df3-943b-a67d568ff472,PASS,1,0 days 00:00:00.340000,0 days 00:00:02.869454,alive,768,768,99174,0.499564,...,0.48318,3468,KICK_OFF,RIGHT_FOOT,None,None,None,None,None,None


In [8]:
# Display dataset info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3415 entries, 0 to 3414
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   event_id            3415 non-null   object         
 1   event_type          3415 non-null   object         
 2   period_id           3415 non-null   int64          
 3   timestamp           3415 non-null   timedelta64[ns]
 4   end_timestamp       1676 non-null   timedelta64[ns]
 5   ball_state          3415 non-null   object         
 6   ball_owning_team    3415 non-null   object         
 7   team_id             3415 non-null   object         
 8   player_id           3400 non-null   object         
 9   coordinates_x       3389 non-null   float64        
 10  coordinates_y       3389 non-null   float64        
 11  result              1844 non-null   object         
 12  success             1844 non-null   object         
 13  end_coordinates_x   1701 non-null

## Filter columns from the dataset

In [9]:
# Filter columns from the dataset
filtered_df = dataset.to_df(
    "player_id",
    "player",
    "team_id",
    "team",
    "event_id",
    "event_type",
    "result",
    "success",
    "body_part_type",
    "pass_type",
    "duel_type",
    "set_piece_type",
    "goalkeeper_type",
    "card_type",
    "coordinates_x",
    "coordinates_y",
    "time",
)

In [10]:
# Create a mapping for dtypes of all columns
dtype_mapping = {
    "player_id": "Int64",
    "player": "string",
    "team_id": "Int64",
    "team": "string",
    "event_id": "string",
    "event_type": "category",
    "result": "category",
    "success": "boolean",
    "body_part_type": "category",
    "pass_type": "category",
    "duel_type": "category",
    "set_piece_type": "category",
    "goalkeeper_type": "category",
    "card_type": "category",
    "coordinates_x": "Float64",
    "coordinates_y": "Float64",
    "time": "string",
}

# Convert the type of all DataFrame columns
filtered_df = filtered_df.astype(dtype_mapping)

# Verify changes
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3415 entries, 0 to 3414
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   player_id        3400 non-null   Int64   
 1   player           3400 non-null   string  
 2   team_id          3415 non-null   Int64   
 3   team             3415 non-null   string  
 4   event_id         3415 non-null   string  
 5   event_type       3415 non-null   category
 6   result           1844 non-null   category
 7   success          1844 non-null   boolean 
 8   body_part_type   938 non-null    category
 9   pass_type        227 non-null    category
 10  duel_type        109 non-null    category
 11  set_piece_type   82 non-null     category
 12  goalkeeper_type  11 non-null     category
 13  card_type        8 non-null      category
 14  coordinates_x    3389 non-null   Float64 
 15  coordinates_y    3389 non-null   Float64 
 16  time             3415 non-null   string  


In [11]:
# Display DataFrame
filtered_df.head()

,player_id,player,team_id,team,event_id,event_type,result,success,body_part_type,pass_type,duel_type,set_piece_type,goalkeeper_type,card_type,coordinates_x,coordinates_y,time
0,<NA>,<NA>,772,Spain,50aa204f-5d65-4145-8597-5d5628fb7898,GENERIC:Starting XI,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
1,<NA>,<NA>,768,England,a279cbee-9ab3-4cfb-9c51-27cacc1bf2a2,GENERIC:Starting XI,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
2,<NA>,<NA>,768,England,d2126e70-9f04-4bb7-ba2b-9377836d1757,GENERIC:Half Start,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
3,<NA>,<NA>,772,Spain,54d78bfa-4146-42bd-acdc-97bcd393dd81,GENERIC:Half Start,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
4,99174,Kobbie Mainoo,768,England,152820f0-6ca9-4df3-943b-a67d568ff472,PASS,COMPLETE,True,RIGHT_FOOT,NaN,NaN,KICK_OFF,NaN,NaN,0.499564,0.499327,P1T00:00


## Group events by player

In [12]:
# Filter DataFrame to only keeps rows related to player events
players_df = filtered_df[filtered_df["player_id"].notna()]
players_df.head()

,player_id,player,team_id,team,event_id,event_type,result,success,body_part_type,pass_type,duel_type,set_piece_type,goalkeeper_type,card_type,coordinates_x,coordinates_y,time
4,99174,Kobbie Mainoo,768,England,152820f0-6ca9-4df3-943b-a67d568ff472,PASS,COMPLETE,True,RIGHT_FOOT,NaN,NaN,KICK_OFF,NaN,NaN,0.499564,0.499327,P1T00:00
5,3468,Jordan Pickford,768,England,d64668c7-747c-4a7d-912c-e1c3ff357a67,GENERIC:Ball Receipt*,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,0.21834,0.48318,P1T00:03
6,3468,Jordan Pickford,768,England,9c107df3-a3c8-4ad5-bc35-00214087a105,CARRY,COMPLETE,True,NaN,NaN,NaN,NaN,NaN,NaN,0.21834,0.48318,P1T00:03
7,3468,Jordan Pickford,768,England,237201b8-aef8-4823-b282-e82875795c07,PASS,OUT,False,LEFT_FOOT,LONG_BALL,NaN,NaN,NaN,NaN,0.244381,0.386189,P1T00:05
8,22084,Bukayo Saka,768,England,c979e198-edc1-4f22-851a-26cedb6474cf,GENERIC:Ball Receipt*,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,0.798231,0.721654,P1T00:10


In [13]:
# Count events by player
event_count_by_player_df = players_df.groupby(["player", "event_type"]).size().unstack()

# Remove columns axis name and convert "player" from index to a column
event_count_by_player_df = event_count_by_player_df.rename_axis(None, axis="columns").reset_index()

# Display the event count by player
event_count_by_player_df.head()

,player,BALL_OUT,CARD,CARRY,CLEARANCE,DUEL,FORMATION_CHANGE,FOUL_COMMITTED,GENERIC:Ball Receipt*,GENERIC:Block,...,GENERIC:Starting XI,GOALKEEPER,INTERCEPTION,MISCONTROL,PASS,PRESSURE,RECOVERY,SHOT,SUBSTITUTION,TAKE_ON
0,Aymeric Laporte,2,0,71,7,7,0,0,74,2,...,0,0,0,0,83,2,1,1,0,0
1,Bukayo Saka,4,0,29,1,3,0,2,32,2,...,0,0,0,0,24,17,3,0,0,1
2,Cole Palmer,1,0,8,1,1,0,0,7,0,...,0,0,0,0,7,7,2,1,0,1
3,Daniel Carvajal Ramos,4,0,50,2,8,0,1,54,3,...,0,0,2,0,73,15,5,0,0,2
4,Daniel Olmo Carvajal,1,1,32,0,3,0,1,40,2,...,0,0,1,2,32,26,3,2,0,1


In [14]:
# Display DataFrame info
event_count_by_player_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 30 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   player                     29 non-null     string
 1   BALL_OUT                   29 non-null     int64 
 2   CARD                       29 non-null     int64 
 3   CARRY                      29 non-null     int64 
 4   CLEARANCE                  29 non-null     int64 
 5   DUEL                       29 non-null     int64 
 6   FORMATION_CHANGE           29 non-null     int64 
 7   FOUL_COMMITTED             29 non-null     int64 
 8   GENERIC:Ball Receipt*      29 non-null     int64 
 9   GENERIC:Block              29 non-null     int64 
 10  GENERIC:Dispossessed       29 non-null     int64 
 11  GENERIC:Dribbled Past      29 non-null     int64 
 12  GENERIC:Error              29 non-null     int64 
 13  GENERIC:Foul Won           29 non-null     int64 
 14  GENERIC:Goal

## Combine Events with Result and Success

In [15]:
# Create comprehensive player statistics with success ratios
def create_player_statistics(df):
    """Create detailed player statistics including success ratios and sub-types"""

    # Start with basic event counts
    basic_events = df.groupby(["player", "event_type"]).size().unstack(fill_value=0)

    # Events that have success ratios - we'll exclude these from basic events
    events_with_results = ["pass", "shot", "take_on", "carry", "interception", "duel"]

    # Remove events that have ratios from basic_events to avoid duplication
    basic_events = basic_events.drop(columns=[col for col in events_with_results if col in basic_events.columns])

    # Create success ratio format for events with results
    success_ratios = {}

    for event in events_with_results:
        if event in df["event_type"].values:
            event_data = df[df["event_type"] == event]

            # Count total and successful for each player
            total_counts = event_data.groupby("player").size()
            success_counts = event_data[event_data["success"] == True].groupby("player").size()

            # Create ratio format "successful/total"
            ratio_series = total_counts.index.map(
                lambda player: f"{success_counts.get(player, 0)}/{total_counts.get(player, 0)}"
            )

            success_ratios[f"{event}_ratio"] = pd.Series(ratio_series.values, index=total_counts.index)

    # Add sub-type details (body_part, pass_type, etc.)
    subtype_columns = ["body_part_type", "pass_type", "duel_type", "set_piece_type", "goalkeeper_type", "card_type"]

    subtype_stats = {}
    for col in subtype_columns:
        if col in df.columns:
            # Convert to lowercase before grouping
            df_copy = df.copy()
            df_copy[col] = df_copy[col].astype(str).str.lower()

            # Count occurrences of each subtype per player
            subtype_counts = (
                df_copy[df_copy[col].notna() & (df_copy[col] != "nan")]
                .groupby(["player", col])
                .size()
                .unstack(fill_value=0)
            )

            # Rename columns to include the type prefix and convert to int
            subtype_counts.columns = [f"{col}_{subcol}" for subcol in subtype_counts.columns]
            subtype_counts = subtype_counts.astype(int)
            subtype_stats[col] = subtype_counts

    return basic_events, success_ratios, subtype_stats


# Apply the function
basic_events, success_ratios, subtype_stats = create_player_statistics(players_df)

print("Basic events shape:", basic_events.shape)
print("Success ratios created:", list(success_ratios.keys()))
print("Subtype categories:", list(subtype_stats.keys()))

Basic events shape: (29, 29)
Success ratios created: []
Subtype categories: ['body_part_type', 'pass_type', 'duel_type', 'set_piece_type', 'goalkeeper_type', 'card_type']


In [16]:
# Create DataFrame with player-team mapping
player_team_mapping = players_df[["player", "team"]].drop_duplicates().reset_index(drop=True)
player_team_mapping.head()

,player,team
0,Kobbie Mainoo,England
1,Jordan Pickford,England
2,Bukayo Saka,England
3,Unai Simón Mendibil,Spain
4,Robin Aime Robert Le Normand,Spain


In [17]:
# Merge all statistics into final comprehensive table
def merge_comprehensive_stats(basic_events, success_ratios, subtype_stats, team_mapping):
    """Merge basic events, success ratios, and subtypes into one comprehensive DataFrame"""

    # Start with basic events and reset index to make player a column
    final_df = basic_events.reset_index()

    # Add team information right after player
    final_df = final_df.merge(team_mapping, on="player", how="left")

    # Reorder to have player, team, then basic events
    basic_cols = [col for col in final_df.columns if col not in ["player", "team"]]
    final_df = final_df[["player", "team"] + basic_cols]

    # Add success ratios
    for ratio_name, ratio_series in success_ratios.items():
        ratio_df = ratio_series.reset_index()
        ratio_df.columns = ["player", ratio_name]
        final_df = final_df.merge(ratio_df, on="player", how="left")

    # Add subtype statistics
    for subtype_name, subtype_df in subtype_stats.items():
        if not subtype_df.empty:
            subtype_reset = subtype_df.reset_index()
            final_df = final_df.merge(subtype_reset, on="player", how="left")

    # Fill NaN values - use 0 for numeric columns and "0/0" for ratios
    ratio_columns = [col for col in final_df.columns if col.endswith("_ratio")]

    # Fill numeric columns with 0 and convert to int where appropriate
    numeric_cols = final_df.select_dtypes(include=["float64", "int64"]).columns
    final_df[numeric_cols] = final_df[numeric_cols].fillna(0).astype(int)

    # Fill ratio columns with "0/0"
    for col in ratio_columns:
        final_df[col] = final_df[col].fillna("0/0")

    return final_df


# Create the comprehensive statistics table
comprehensive_stats = merge_comprehensive_stats(basic_events, success_ratios, subtype_stats, player_team_mapping)

print("Comprehensive player statistics:")
print(f"Shape: {comprehensive_stats.shape}")
print(f"Columns: {list(comprehensive_stats.columns)}")
print("\nFirst 3 players:")
comprehensive_stats.head()

Comprehensive player statistics:
Shape: (29, 57)
Columns: ['player', 'team', 'BALL_OUT', 'CARD', 'CARRY', 'CLEARANCE', 'DUEL', 'FORMATION_CHANGE', 'FOUL_COMMITTED', 'GENERIC:Ball Receipt*', 'GENERIC:Block', 'GENERIC:Dispossessed', 'GENERIC:Dribbled Past', 'GENERIC:Error', 'GENERIC:Foul Won', 'GENERIC:Goal Keeper', 'GENERIC:Half End', 'GENERIC:Half Start', 'GENERIC:Injury Stoppage', 'GENERIC:Referee Ball-Drop', 'GENERIC:Shield', 'GENERIC:Starting XI', 'GOALKEEPER', 'INTERCEPTION', 'MISCONTROL', 'PASS', 'PRESSURE', 'RECOVERY', 'SHOT', 'SUBSTITUTION', 'TAKE_ON', 'body_part_type_both_hands', 'body_part_type_head', 'body_part_type_keeper_arm', 'body_part_type_left_foot', 'body_part_type_no_touch', 'body_part_type_other', 'body_part_type_right_foot', 'body_part_type_right_hand', 'pass_type_cross', 'pass_type_hand_pass', 'pass_type_head_pass', 'pass_type_high_pass', 'pass_type_long_ball', 'pass_type_shot_assist', 'duel_type_aerial', 'duel_type_ground', 'duel_type_loose_ball', 'set_piece_type_

,player,team,BALL_OUT,CARD,CARRY,CLEARANCE,DUEL,FORMATION_CHANGE,FOUL_COMMITTED,GENERIC:Ball Receipt*,...,duel_type_loose_ball,set_piece_type_corner_kick,set_piece_type_free_kick,set_piece_type_goal_kick,set_piece_type_kick_off,set_piece_type_throw_in,goalkeeper_type_claim,goalkeeper_type_punch,goalkeeper_type_save,card_type_first_yellow
0,Aymeric Laporte,Spain,2,0,71,7,7,0,0,74,...,0,0,1,2,0,0,0,0,0,0
1,Bukayo Saka,England,4,0,29,1,3,0,2,32,...,0,0,0,0,0,0,0,0,0,0
2,Cole Palmer,England,1,0,8,1,1,0,0,7,...,0,1,1,0,0,0,0,0,0,0
3,Daniel Carvajal Ramos,Spain,4,0,50,2,8,0,1,54,...,0,0,0,0,0,14,0,0,0,0
4,Daniel Olmo Carvajal,Spain,1,1,32,0,3,0,1,40,...,0,0,0,0,0,0,0,0,0,2


## Constants

In [18]:
# COEFFICIENTS FOR THE INDEX

## Subindex 1: Model coefficients
MODEL_COEFFICIENTS = {
    "crosses": 0.519,
    "dribbles": 0.118,
    "passes": 0.034,
    "opp_interceptions": -0.024,
    "opp_yellows": 0.253,
    "opp_reds": 1.023,
    "opp_tackle_win_ratio": -0.170,
    "opp_clearances": -0.017,
    "constant": 6.463,
}

## Subindex 2
POINTS_FOR_WIN = 3
POINTS_FOR_DRAW = 1
POINTS_FOR_LOSS = 0

## Subindex 3
POINTS_PER_GAME = 1.34

## Subindex 4
POINTS_PER_GOAL = 1.039

## Subindex 5
POINTS_PER_ASSIST = 1.039

## Subindex 6
POINTS_PER_CLEAN_SHEET_GOALKEEPER = 0.585
POINTS_PER_CLEAN_SHEET_DEFENDER = 0.364
POINTS_PER_CLEAN_SHEET_MIDFIELDER = 0.150
POINTS_PER_CLEAN_SHEET_STRIKER = 0.071

In [19]:
# Necessary variables for the index with dummy values
position = "ST"
minutes_played = 90
goals = 2
assists = 1

home_goals = 4
away_goals = 2

team_minutes = 990
clean_sheets = 0

stats = {
    "crosses": 3,
    "dribbles": 1,
    "passes": 20,
    "opp_interceptions": 3,
    "opp_yellows": 1,
    "opp_reds": 0,
    "opp_tackle_win_ratio": 0.2,
    "opp_clearances": 1,
}

## The Index

### Subindex 1: Modelling Match Outcome

In [20]:
index_1 = MODEL_COEFFICIENTS["constant"]
for action, coeff in MODEL_COEFFICIENTS.items():
    if action != "constant":
        count = stats.get(action, 0)
        index_1 += coeff * count

### Subindex 2: Points-Sharing Index

In [21]:
if home_goals > away_goals:
    index_2 = (minutes_played / team_minutes) * POINTS_FOR_WIN
elif home_goals == away_goals:
    index_2 = (minutes_played / team_minutes) * POINTS_FOR_DRAW
else:  # home_goals < away_goals
    index_2 = (minutes_played / team_minutes) * POINTS_FOR_LOSS

### Subindex 3: Appearance Index

In [22]:
index_3 = (minutes_played / team_minutes) * POINTS_PER_GAME

### Subindex 4: Goal-Scoring Index

In [23]:
index_4 = goals * POINTS_PER_GOAL

### Subindex 5: Assists Index

In [24]:
index_5 = assists * POINTS_PER_ASSIST

### Subindex 6: Clean-Sheets Index

In [25]:
if clean_sheets:
    if position == "goalkeeper":
        index_6 = POINTS_PER_CLEAN_SHEET_GOALKEEPER
    elif position == "defender":
        index_6 = POINTS_PER_CLEAN_SHEET_DEFENDER
    elif position == "midfielder":
        index_6 = POINTS_PER_CLEAN_SHEET_MIDFIELDER
    elif position == "striker":
        index_6 = POINTS_PER_CLEAN_SHEET_STRIKER
else:
    index_6 = 0

## Final Index

In [26]:
final_index = 100 * (
    0.25 * index_1 + 0.375 * index_2 + 0.125 * index_3 + 0.125 * index_4 + 0.0625 * index_5 + 0.0625 * index_6
)

In [27]:
round(final_index, 2)

267.92

## Extract metrics for PPI calculation

In [28]:
minutes_played_dataset = dataset.aggregate("minutes_played")

In [29]:
from kloppy.domain import EventDataset


# Extract individual player metrics needed for PPI
def extract_player_metrics(dataset: EventDataset, df: pd.DataFrame, minutes_data: list):
    """Extract individual player metrics: crosses, dribbles, passes, goals, assists, minutes"""

    # Define dictionary to hold player metrics
    player_metrics = {}

    # Get minutes played for each player
    minutes_dict = {str(item.player): math.ceil(item.duration.total_seconds() / 60) for item in minutes_data}

    # Fix assists calculation - look for SHOT_ASSIST before goals
    def calculate_assists_for_player(player_data, all_data):
        """Calculate assists by looking for SHOT_ASSIST events before goals"""

        # Get all goals in chronological order by DataFrame index
        goals = all_data[(all_data["event_type"] == "SHOT") & (all_data["result"] == "GOAL")].sort_index()

        assists_count = 0

        for goal_idx, goal in goals.iterrows():
            # Look for SHOT_ASSIST in the previous 3 events for this player
            for i in range(1, 4):  # Check the 1st, 2nd, and 3rd events before
                prev_idx = goal_idx - i
                if prev_idx < 0:
                    continue
                assist_event = player_data.loc[
                    (player_data.index == prev_idx) & (player_data["event_type"] == "SHOT_ASSIST")
                ]
                if not assist_event.empty:
                    assists_count += 1
                    break  # Found assist for this goal, move to next goal

        return assists_count

    def get_player_position(dataset: EventDataset, player_name: str) -> str | None:
        """Determine main position for a player based on time spent in each position"""
        # Set default position and duration
        player_position = (None, 0)

        # Iterate through dataset to find player's position history
        for team in dataset.metadata.teams:
            for player in team.players:
                # Only get position for the specified player
                if player.name == player_name:
                    for start_time, end_time, position in player.positions.ranges():
                        duration = (end_time - start_time).total_seconds()

                        # Navigate to obtain the position group
                        while position.parent is not None:
                            position = position.parent

                        # Compare and update dictionary if this position has longer duration
                        prev_position, prev_duration = player_position
                        if (position.name, duration) != (prev_position, prev_duration) and duration > prev_duration:
                            player_position = (position.name, duration)

        # Return only the position name
        return player_position[0]

    # Count specific events per player
    for player in df["player"].unique():
        # Filter df for the player
        player_data = df[df["player"] == player]

        # Individual metrics
        minutes_played = minutes_dict.get(player, 0)
        position = get_player_position(dataset, player)
        goals = len(player_data[player_data["event_type"] == "SHOT"][player_data["result"] == "GOAL"])
        assists = calculate_assists_for_player(player_data, df)
        crosses = len(player_data[player_data["event_type"] == "PASS"][player_data["pass_type"] == "CROSS"])
        dribbles = len(player_data[player_data["event_type"] == "TAKE_ON"])
        passes = len(player_data[player_data["event_type"] == "PASS"])

        # Store metrics in dictionary
        player_metrics[player] = {
            "team": player_data["team"].iloc[0],
            "minutes_played": minutes_played,
            "position": position,
            "goals": goals,
            "assists": assists,
            "crosses": crosses,
            "dribbles": dribbles,
            "passes": passes,
        }

    return player_metrics


# Extract team-level metrics
def extract_team_metrics(df: pd.DataFrame, minutes_played_dataset: list):
    """Extract team metrics: interceptions, yellows, reds, tackle wins, clearances"""

    # Define dictionary to store team metrics
    team_metrics = {}

    for team in df["team"].unique():
        team_data = df[df["team"] == team]

        # Interceptions
        team_interceptions = len(team_data[team_data["event_type"] == "INTERCEPTION"])

        # Yellow and red cards
        team_yellow_cards = len(team_data[team_data["event_type"] == "CARD"][team_data["card_type"] == "FIRST_YELLOW"])
        team_red_cards = len(
            team_data[team_data["event_type"] == "CARD"][team_data["card_type"].isin(["RED", "SECOND_YELLOW"])]
        )

        # Tackle win ratio calculation (successful duels / total duels)
        duels = team_data[team_data["event_type"] == "DUEL"]

        total_duels = len(duels)
        won_duels = len(duels[duels["success"]])
        team_tackle_win_ratio = round(won_duels / total_duels, 2) if total_duels > 0 else 0

        # Clearances
        team_clearances = len(team_data[team_data["event_type"] == "CLEARANCE"])

        # Goals scored
        goals_scored = len(team_data[team_data["event_type"] == "SHOT"][team_data["result"] == "GOAL"])

        # Team minutes - calculate from minutes_played_dataset
        team_minutes = sum(
            [
                math.ceil(item.duration.total_seconds() / 60)
                for item in minutes_played_dataset
                if str(item.player) in df[df["team"] == team]["player"].values
            ]
        )

        # Store metrics in dictionary
        team_metrics[team] = {
            "interceptions": team_interceptions,
            "yellow_cards": team_yellow_cards,
            "red_cards": team_red_cards,
            "tackle_win_ratio": team_tackle_win_ratio,
            "clearances": team_clearances,
            "goals_scored": goals_scored,
            "team_minutes": team_minutes,
        }

    return team_metrics


# Extract metrics
player_metrics = extract_player_metrics(dataset, players_df, minutes_played_dataset)
team_metrics = extract_team_metrics(players_df, minutes_played_dataset)

print("Player metrics extracted for", len(player_metrics), "players")
print("Team metrics extracted for", len(team_metrics), "teams")

print("\nPlayer metrics:")
for k, v in player_metrics.items():
    print(f"Player: {k}")
    print(f"        {v}")

print("\nTeam metrics:")
for k, v in team_metrics.items():
    print(f"Team: {k}")
    print(f"      {v}")

Player metrics extracted for 29 players
Team metrics extracted for 2 teams

Player metrics:
Player: Kobbie Mainoo
        {'team': 'England', 'minutes_played': 72, 'position': 'Midfielder', 'goals': 0, 'assists': 0, 'crosses': 0, 'dribbles': 3, 'passes': 18}
Player: Jordan Pickford
        {'team': 'England', 'minutes_played': 97, 'position': 'Goalkeeper', 'goals': 0, 'assists': 0, 'crosses': 0, 'dribbles': 0, 'passes': 42}
Player: Bukayo Saka
        {'team': 'England', 'minutes_played': 97, 'position': 'Midfielder', 'goals': 0, 'assists': 0, 'crosses': 2, 'dribbles': 1, 'passes': 24}
Player: Unai Simón Mendibil
        {'team': 'Spain', 'minutes_played': 97, 'position': 'Goalkeeper', 'goals': 0, 'assists': 0, 'crosses': 0, 'dribbles': 0, 'passes': 40}
Player: Robin Aime Robert Le Normand
        {'team': 'Spain', 'minutes_played': 85, 'position': 'Defender', 'goals': 0, 'assists': 0, 'crosses': 0, 'dribbles': 0, 'passes': 84}
Player: Daniel Carvajal Ramos
        {'team': 'Spain', 'm

In [30]:
# Calculate PPI for each player
def calculate_ppi_for_players(player_metrics, team_metrics, model_coefficients):
    """Calculate EA Sports PPI for each player"""

    player_ppi_scores = []

    # Get team totals for calculations
    total_team_minutes = {}
    for team, metrics in team_metrics.items():
        # Sum all player minutes for this team
        team_player_minutes = sum([pm["minutes_played"] for pm in player_metrics.values() if pm["team"] == team])
        total_team_minutes[team] = team_player_minutes

    # Determine match result (simplified - using goals from first two teams)
    teams = list(team_metrics.keys())
    if len(teams) >= 2:
        home_goals = team_metrics[teams[0]]["goals_scored"]
        away_goals = team_metrics[teams[1]]["goals_scored"]
    else:
        home_goals = away_goals = 0

    for player, metrics in player_metrics.items():
        if metrics["team"] is None:
            continue

        team = metrics["team"]
        position = metrics["position"]
        team_stats = team_metrics[team]
        team_minutes = total_team_minutes[team]

        # Subindex 1: Modelling Match Outcome
        index_1 = model_coefficients["constant"]
        index_1 += model_coefficients["crosses"] * metrics["crosses"]
        index_1 += model_coefficients["dribbles"] * metrics["dribbles"]
        index_1 += model_coefficients["passes"] * metrics["passes"]
        index_1 += model_coefficients["opp_interceptions"] * team_stats["interceptions"]
        index_1 += model_coefficients["opp_yellows"] * team_stats["yellow_cards"]
        index_1 += model_coefficients["opp_reds"] * team_stats["red_cards"]
        index_1 += model_coefficients["opp_tackle_win_ratio"] * team_stats["tackle_win_ratio"]
        index_1 += model_coefficients["opp_clearances"] * team_stats["clearances"]

        # Subindex 2: Points-Sharing Index
        minutes_ratio = metrics["minutes_played"] / team_minutes if team_minutes > 0 else 0
        if home_goals > away_goals:
            index_2 = minutes_ratio * POINTS_FOR_WIN
        elif home_goals == away_goals:
            index_2 = minutes_ratio * POINTS_FOR_DRAW
        else:
            index_2 = minutes_ratio * POINTS_FOR_LOSS

        # Subindex 3: Appearance Index
        index_3 = minutes_ratio * POINTS_PER_GAME

        # Subindex 4: Goal-Scoring Index
        index_4 = metrics["goals"] * POINTS_PER_GOAL

        # Subindex 5: Assists Index
        index_5 = metrics["assists"] * POINTS_PER_ASSIST

        # Subindex 6: Clean-Sheets Index (using actual positions)
        # Determine if team kept clean sheet (opponent scored 0 goals)
        opponent_goals = away_goals if team == teams[0] else home_goals
        clean_sheets = 1 if opponent_goals == 0 else 0

        if clean_sheets:
            if position == "goalkeeper":
                index_6 = POINTS_PER_CLEAN_SHEET_GOALKEEPER
            elif position == "defender":
                index_6 = POINTS_PER_CLEAN_SHEET_DEFENDER
            elif position == "midfielder":
                index_6 = POINTS_PER_CLEAN_SHEET_MIDFIELDER
            elif position == "striker":
                index_6 = POINTS_PER_CLEAN_SHEET_STRIKER
            else:
                index_6 = POINTS_PER_CLEAN_SHEET_MIDFIELDER  # Default
        else:
            index_6 = 0

        # Final PPI calculation
        final_ppi = 100 * (
            0.25 * index_1 + 0.375 * index_2 + 0.125 * index_3 + 0.125 * index_4 + 0.0625 * index_5 + 0.0625 * index_6
        )

        player_ppi_scores.append(
            {
                "player": player,
                "team": team,
                "position": position,
                "ppi_score": round(final_ppi, 2),
                "minutes_played": metrics["minutes_played"],
                "goals": metrics["goals"],
                "assists": metrics["assists"],
                "crosses": metrics["crosses"],
                "dribbles": metrics["dribbles"],
                "passes": metrics["passes"],
            }
        )

    return player_ppi_scores


# Calculate PPI scores
ppi_scores = calculate_ppi_for_players(player_metrics, team_metrics, MODEL_COEFFICIENTS)

# Convert to DataFrame and rank players
ppi_df = pd.DataFrame(ppi_scores)
ppi_df = ppi_df.sort_values("ppi_score", ascending=False).reset_index(drop=True)
ppi_df["rank"] = range(1, len(ppi_df) + 1)

# Display all players ranked by PPI
ppi_df[["rank", "player", "team", "position", "ppi_score", "minutes_played", "goals", "assists"]]

,rank,player,team,position,ppi_score,minutes_played,goals,assists
0,1,Nicholas Williams Arthuer,Spain,Midfielder,234.64,97,1,0
1,2,Robin Aime Robert Le Normand,Spain,Defender,227.69,85,0,0
2,3,Aymeric Laporte,Spain,Defender,227.03,97,0,0
3,4,Daniel Carvajal Ramos,Spain,Defender,224.43,97,0,0
4,5,Kyle Walker,England,Defender,219.66,97,0,0
5,6,Bukayo Saka,England,Midfielder,214.38,97,0,0
6,7,Fabián Ruiz Peña,Spain,Midfielder,213.83,97,0,0
7,8,Declan Rice,England,Midfielder,206.68,97,0,0
8,9,Jordan Pickford,England,Goalkeeper,200.78,97,0,0
9,10,Jude Bellingham,England,Midfielder,197.78,97,0,0
